# Semantic Scholar

Code for accessing the [semantic scholar database](https://www.semanticscholar.org)

In [ ]:
#| default_exp scholar

In [ ]:
#| hide
#| export
import pandas as pd
import numpy as np
import time
import re
from biobib.doc import Doc, NumbersDoc
from jinja2 import Environment, FileSystemLoader, Template
import copy
from nbdev import show_doc
from fastcore.test import *
from fastcore.test import *
from fastcore.basics import *
from fastcore.foundation import *
import os
from dotenv import load_dotenv
from biobib.website import authors as site_authors
from biobib.website import normalize_name
from fuzzywuzzy import fuzz
import re 
from pdf2image import convert_from_path
from PIL import Image
import PyPDF2


ModuleNotFoundError: No module named 'biobib'

## Load the API key using dot_env

In [ ]:
#| hide
#| export
# Load environment variables from .env file
load_dotenv()

True

In [ ]:
# Access the API key using os.getenv
API_KEY = os.getenv("SCHOLAR_API_KEY")

In [ ]:
from biobib.Publications import Publications

doc = NumbersDoc("CV.numbers")
pubs = Publications(doc)

In [ ]:
missing_pubs = pubs.df[pubs.df['Web'] == 0].copy()

In [ ]:
missing_pubs

,NUM,YEAR,Type,TITLE,PUBLISHER,#,CODE,New?,Kind,Undergrad Author,...,Resources,Software,Supervision,Validation,Visualization,Writing - Original Draft,Writing - review & editing,Web,authors,editors
117,118.0,2020.0,P,Geographic Object-Based Image Analysis Framewo...,Remote Sensing,F,P20F,N,RA,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,"Ribeiro, F., Roberts, D.A., Hess, L.L., Davis,...",
144,145.0,2024.0,P,Field-scale crop water consumption estimates r...,Nature Communications,C,R23C,Y,RA,None,...,1.0,NaN,1.0,NaN,NaN,NaN,1.0,0.0,"Boser, A., Caylor, K.K., Larsen, A., Pascolini...",
145,146.0,2024.0,P,Groundwater for Ecosystems: Establishing ecolo...,Nature Water,F,F23C,Y,RA,None,...,NaN,NaN,1.0,NaN,NaN,NaN,1.0,0.0,"Rohde, M.M, Stella, J., Singer, M.B., Roberts,...",


### Get publication data based on doi

In [ ]:
from semanticscholar import SemanticScholar
sch = SemanticScholar(api_key=API_KEY)

In [ ]:
#| export
import requests

def get_pub(doi, api_key=API_KEY):
    rsp = requests.get(f'https://api.semanticscholar.org/graph/v1/paper/{doi}',
        headers = {'X-API-KEY': api_key})
    rsp.raise_for_status()
    return rsp.json()

def get_citations(paper_id):
    edges = get_citation_edges(url=f'https://api.semanticscholar.org/graph/v1/paper/{paper_id}/citations',
                               headers={'X-API-KEY': S2_API_KEY},
                               params={'fields': 'title,authors'})
    return list(edge['citingPaper'] for edge in edges)


def get_references(paper_id):
    edges = get_citation_edges(url=f'https://api.semanticscholar.org/graph/v1/paper/{paper_id}/references',
                               headers={'X-API-KEY': S2_API_KEY},
                               params={'fields': 'title,authors'})
    return list(edge['citedPaper'] for edge in edges)

In [ ]:
#| export
jekyll_env = Environment(
    extensions=['jinja2.ext.do'],
    block_start_string='\BLOCK{',
    block_end_string='}',
    variable_start_string='\VAR{',
    variable_end_string='}',
    comment_start_string='\#{',
    comment_end_string='}',
    line_statement_prefix='%%',
    line_comment_prefix='%#',
    trim_blocks=True,
    autoescape=False,
    loader=FileSystemLoader('templates'))

paper_template = r'''---
author: \VAR{pub.author}
date: \VAR{pub.date}
id: \VAR{pub.id}
year: '\VAR{pub.year}'
title: "\VAR{pub.title}"
doi: \VAR{pub.doi}
excerpt: "\VAR{pub.excerpt}"
header:
   teaser: \VAR{pub.teaser_image}
portfolio-item-category:
   - publications
portfolio-item-tag:
    - '\VAR{pub.year}'
    - \VAR{pub.journal}

author-tags:
\BLOCK{ for author in pub.site_authors }
    - \VAR{author}
\BLOCK{ endfor }

---

![ first page ]( {{ "\VAR{pub.figure_image}" | absolute_url }} ){:class="img-responsive" width="50%" .align-right}

> \VAR{pub.citation}

**Abstract**: \VAR{pub.abstract}

[Go to the Article](https://www.doi.org/\VAR{pub.doi}){: .btn .btn--success}
'''

In [ ]:
from jinja2 import Template, Environment
import os
import random

class Paper:
    def __init__(self, doi, template):
        self.doi = doi
        self.template_str = template
        self.data = self._fetch_data()
        self.date = self.pub_date()
        self.title = self.data.title
        self.journal = self.data.venue
        self.year = self.data.year
        self.id = self._generate_id()
        self.citation = self._format_citation()
        self.excerpt = self._format_citation(et_al=True)
        self.abstract = self.data.abstract
        self.doi = self.data.externalIds['DOI']
        self.site_authors = self._site_authors()
        self.author = self.get_author()
        self.filename = self._filename(ext='md')
        self.teaser_image = self.teaser()
        self.figure_image = self.figure()
    
    @classmethod
    def make_paper(cls, doi):
        try:
            return cls(doi, paper_template)
        except Exception as e:
            print(f"Error making paper for {doi}: {e}")

    def pub_date(self):
        if self.data.publicationDate:
            return self.data.publicationDate.strftime('%Y-%m-%d %H:%M:%S')
        else:
            return f'{self.data.year}-06-01 00:00:00'
    
    def _filename(self, ext=None):
        if ext:
            return f"{self.author_last_name()}{self.year}_{self.id}.{ext}"
        else:
            return f"{self.author_last_name()}{self.year}_{self.id}"


    def figure(self):
        return f"assets/images/publications/{self._filename()}_figure.png"
    
    def teaser(self):
        return f"assets/images/publications/{self._filename()}.png"
    
    def get_author(self):
        return self.site_authors[0]
           
    def author_last_name(self):
        return self.author.split()[-1]

    def get_authors(self):
        """Get the author name from the data."""
        return self.data.authors

    def _generate_id(self):
        return random.randint(1000, 9999)

    def _fetch_data(self):
        """Private method to fetch paper data using the DOI."""
        return sch.get_paper(self.doi)

    def render_markdown(self, env=jekyll_env):
        """Make markdown files for publications.

        Parameters
        ----------
        output_dir : str
            The directory to save the markdown file to.
        
        """
        _template = env.from_string(self.template_str)
        markdown = _template.render(pub=self)
        return markdown

    def save_markdown(self, output_dir=os.getenv('WEB_DIR')):
        """Save the markdown file to the output directory."""
        markdown = self.render_markdown()
        with open(os.path.join(output_dir, '_publications',  self.filename), 'w') as f:
            f.write(markdown)
    
    def _site_authors(self,site_authors=site_authors, threshold=80):
        """ Return a list of paper authors that match the site authors list.
        """
        matched_authors = []

        # For each API author, check if they are in the site authors list
        # For each API author, check if they are in the site authors list
        for api_author in self.get_authors():
            max_score = 0
            best_match = None

            # Consider all aliases for the API author
            aliases = [api_author['name']] + (api_author['aliases'] or [])
            for alias in aliases:
                normalized_api_alias = normalize_name(alias)

                # Check for a match in the site authors list
                for site_author_name, site_author_details in site_authors.items():
                    score = fuzz.partial_ratio(normalized_api_alias, site_author_details['normalized_name'])
                    if score > max_score:
                        max_score = score
                        best_match = site_author_name

            # Check if the highest scoring match is above your threshold
            if max_score > threshold:  # Adjust this threshold as needed
                matched_authors.append(best_match)

        matched_authors.append('Kelly Caylor')
        return matched_authors


    def extract_first_page(self, pdf_path, output_dir, output_filename, max_dimension=640):
        """
        Extracts the first page from a PDF file and saves it as a PNG file.

        :param pdf_path: Path to the PDF file
        :param output_dir: Directory to save the output PNG file
        :param output_filename: Name of the output PNG file
        :param max_dimension: The maximum dimension (width or height) of the output image
        """
        # Convert the first page of the PDF to an image
        images = convert_from_path(pdf_path, first_page=1, last_page=1)

        if images:
            image = images[0]
            # Resize the image if necessary
            if image.width > max_dimension or image.height > max_dimension:
                # Maintain aspect ratio
                image.thumbnail((max_dimension, max_dimension), Image.ANTIALIAS)

            # Save the image
            output_path = os.path.join(output_dir, output_filename)
            image.save(output_path, 'PNG')

            print(f"Saved first page as PNG to {output_path}")
        else:
            print("No pages found in the PDF file.")

    def _format_citation(self, et_al=False):
        """
        Format the paper data into an APA-style citation string.

        Returns
        -------
        str
            A formatted citation string in APA style.
        """
        
        # Extracting authors and using only initials for first and middle names
        authors = self.data.authors
        def format_author_name(name):
            parts = name.split()
            initials = ''.join([part[0] + '.' for part in parts[:-1]])
            return f"{parts[-1]}, {initials}"

        if et_al and len(authors) > 2:
            author_str = format_author_name(authors[0]['name']) + " et al."
        else:
            author_str = ', '.join([format_author_name(author['name']) for author in authors[:-1]])
            if len(authors) > 1:
                author_str += f", & {format_author_name(authors[-1]['name'])}"
        
        # Extracting other details
        title = self.data.title
        journal = self.data.venue
        year = self.data.year
        doi = self.data.externalIds['DOI']
        # Formatting the citation string in APA style
        citation = f"{author_str} ({year}). {title}. _{journal}_, doi:{doi}."

        return citation        

In [ ]:
# Usage example:

missing_pubs['Paper'] = missing_pubs['DOI'].apply(Paper.make_paper)

Error making paper for 10.3390/rs12111721: HTTPSConnectionPool(host='partner.semanticscholar.org', port=443): Max retries exceeded with url: /graph/v1/paper/10.3390/rs12111721?&fields=abstract,authors,authors.affiliations,authors.aliases,authors.authorId,authors.citationCount,authors.externalIds,authors.hIndex,authors.homepage,authors.name,authors.paperCount,authors.url,citationCount,citations,citations.abstract,citations.authors,citations.citationCount,citations.corpusId,citations.externalIds,citations.fieldsOfStudy,citations.influentialCitationCount,citations.isOpenAccess,citations.journal,citations.openAccessPdf,citations.paperId,citations.publicationDate,citations.publicationTypes,citations.publicationVenue,citations.referenceCount,citations.s2FieldsOfStudy,citations.title,citations.url,citations.venue,citations.year,corpusId,embedding,externalIds,fieldsOfStudy,influentialCitationCount,isOpenAccess,journal,openAccessPdf,paperId,publicationDate,publicationTypes,publicationVenue,refe

In [ ]:
missing_pubs['Paper'].apply(lambda obj: obj.save_markdown() ) 

AttributeError: 'NoneType' object has no attribute 'save_markdown'

In [ ]:
# | hide
import nbdev
nbdev.nbdev_export()